# N2 — Nettoyage du questionnaire

## Étape 1 — Chargement et exploration

Lecture de l'export CSV de la plateforme de sondage, puis taux de manquants par colonne trié par ordre décroissant, pour repérer les questions peu renseignées (questions filtrées, abandons).

> ⚠️ **Données synthétiques** — ce notebook s'exécute sur des données générées par `generate_dummy_data.py`. Les valeurs, proportions et graphiques affichés sont aléatoires et ne décrivent aucune population réelle.

| | Fichiers |
|---|---|
| **Entrée** | `data/1_Donnees_brutes/base_questionnaire.csv` et `base_questionnaire_brute.csv` |
| **Sortie** | `data/2_Donnees_nettoyees/base_questionnaire_nettoyee.xlsx` |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer

In [2]:
df_quest = pd.read_csv("../data/1_Donnees_brutes/base_questionnaire.csv", encoding="utf-8")

print(f"Dimensions initiales : {df_quest.shape[0]} lignes × {df_quest.shape[1]} colonnes")
print(f"\nManquants par colonne (en %) :")
print((df_quest.isna().sum() / len(df_quest) * 100).round(1).sort_values(ascending=False))

Dimensions initiales : 80 lignes × 49 colonnes

Manquants par colonne (en %) :
À combien estimez-vous le montant de ces envois ?                                         95.0
À quelle fréquence pensez-vous effectuer ces envois ?                                     95.0
Veuillez préciser :.1                                                                     83.8
Veuillez préciser :                                                                       80.0
Quelle est votre relation avec ces personnes ? [Autre]                                    68.8
Quelle est votre relation avec ces personnes ? [Parent]                                   68.8
Quelle est votre relation avec ces personnes ? [Ami]                                      67.5
Quelle est votre relation avec ces personnes ? [Frère / Sœur]                             67.5
À quelle fréquence ?                                                                      56.2
Souhaitez-vous retourner un jour dans votre pays ?                

## Étape 2 — Suppression des colonnes inutiles

Suppression des métadonnées techniques de l'export (identifiant de réponse, horodatage, dernière page vue, langue, graine) et des deux champs de texte libre « Veuillez préciser », non exploitables sans codage manuel. Le suffixe `.1` est ajouté automatiquement par pandas à la lecture, les deux colonnes portant le même intitulé.

In [3]:
colonnes_a_supprimer = [
    "ID de la réponse",
    "Date de soumission",
    "Dernière page",
    "Langue de départ",
    "Tête de série",
    "Veuillez préciser :",
    "Veuillez préciser :.1"
]

df_quest = df_quest.drop(columns=colonnes_a_supprimer)
print(f"Après suppression : {df_quest.shape[1]} colonnes")

Après suppression : 42 colonnes


## Étape 3 — Suppression des questionnaires abandonnés

`dropna(thresh=10)` ne conserve que les lignes comportant au moins 10 réponses renseignées. En dessous de ce seuil, le questionnaire est considéré comme abandonné : la ligne apporterait plus de bruit que d'information à l'imputation.

In [4]:
seuil = 10
# garde les lignes avec au moins 10 réponses non-NaN
df_quest = df_quest.dropna(thresh=seuil)
print(f"Après suppression des lignes vides : {df_quest.shape[0]} réponses")

Après suppression des lignes vides : 63 réponses


## Étape 4 — Harmonisation des noms de colonnes

Les intitulés de questions sont renommés avec les noms de variables de la base d'enquête (`Age`, `Genderf`, `Regionf`, `TrustCourts`...) afin que la concaténation des deux sources (N4) aligne automatiquement les colonnes communes. Les questions propres au questionnaire reçoivent un nom court en snake_case.

In [5]:
renommage = {
    "Quel est votre âge ?": "Age",
    "Quel est votre genre ?": "Genderf",
    "Quel est votre pays d'origine ?": "COUNRY_ID",
    "Dans quel environnement viviez-vous dans votre pays d'origine ?": "Regionf",
    "Quel niveau d'étude avez vous atteint ?": "Educationf",
    "Quelle est votre situation familiale ? ": "Social statusf",
    "Quelle est votre situation professionnelle ?": "Professional situationf",
    "De combien de personnes votre ménage est-il composé ?": "Taille_menage",
    "Êtes-vous la personne de référence concernant les décisions financières de ce ménage ?": "Decideur_financier",
    "Quel est le revenu mensuel moyen de l'ensemble de votre ménage ?": "Revenu",
    "Quelle est la source de ces revenus ? [Agriculture]": "Source_revenu_agriculture",
    "Quelle est la source de ces revenus ? [Commerce]": "Source_revenu_commerce",
    "Quelle est la source de ces revenus ? [Salariat]": "Source_revenu_salariat",
    "Quelle est la source de ces revenus ? [Aide familiale]": "Source_revenu_aide_familiale",
    "Quelle est la source de ces revenus ? [Transferts d'argent de l'étranger]": "Source_revenu_transferts",
    "Quelle est la source de ces revenus ? [Autre]": "Source_revenu_autre",
    "Avez-vous un compte en banque ?": "Bankaccount",
    "Faites-vous des virements ?": "Remit",
    "À quelle fréquence ?": "Frequence_virements",
    "Au cours des 12 derniers mois, avez-vous subi un choc économique ?": "Choc_economique",
    "Avez-vous accès au crédit bancaire ou informel ?": "Acces_credit",
    "Considérez-vous votre pays comme corrompu ?": "Corruption (0  1)",
    "Avez-vous confiance en votre pays ?": "GeneralizedTrsut",
    "Avez-vous confiance dans la justice de votre pays ?": "TrustCourts",
    "Avez-vous confiance dans votre municipalité de votre pays ?": "TrustLocalGovernement",
    "Avez-vous confiance dans le parlement de votre pays ?": "Trust_Parliament",
    "Avez-vous confiance dans votre gouvernement régional ?": "Trust_Regiol_Governement",
    "Avez-vous l'intention de migrer légalement ?": "intention_to_migrate_legal",
    "Avez-vous l'intention de migrer illégalement ?": "intention_to_migrate_illegal",
    "Votre pays d'accueil est-il une étape pour votre destination ?": "Pays_etape",
    "Souhaitez-vous obtenir la nationalité française ?": "Nationalite_francaise",
    "Considérez-vous le mariage blanc ?": "Mariage_blanc",
    "Quelles raisons vous ont poussé à migrer ?": "Raisons_migration",
    "Souhaitez-vous retourner un jour dans votre pays ?": "Retour_pays",
    "Avez-vous des proches en Europe ?": "Proches_europe",
    "Quelle est votre relation avec ces personnes ? [Frère / Sœur]": "Proche_fratrie",
    "Quelle est votre relation avec ces personnes ? [Parent]": "Proche_parent",
    "Quelle est votre relation avec ces personnes ? [Ami]": "Proche_ami",
    "Quelle est votre relation avec ces personnes ? [Autre]": "Proche_autre",
    "Avez-vous l'intention d'envoyer de l'argent à votre famille restée sur place ?": "Intention_envoyer_argent",
    "À combien estimez-vous le montant de ces envois ?": "Montant_envois",
    "À quelle fréquence pensez-vous effectuer ces envois ?": "Frequence_envois"
}

df_quest = df_quest.rename(columns=renommage)
print("Colonnes renommées")
print(df_quest.columns.tolist())

Colonnes renommées
['Age', 'Genderf', 'COUNRY_ID', 'Regionf', 'Educationf', 'Social statusf', 'Professional situationf', 'Taille_menage', 'Decideur_financier', 'Revenu', 'Source_revenu_agriculture', 'Source_revenu_commerce', 'Source_revenu_salariat', 'Source_revenu_aide_familiale', 'Source_revenu_transferts', 'Source_revenu_autre', 'Bankaccount', 'Remit', 'Frequence_virements', 'Choc_economique', 'Acces_credit', 'Corruption (0  1)', 'GeneralizedTrsut', 'TrustCourts', 'TrustLocalGovernement', 'Trust_Parliament', 'Trust_Regiol_Governement', 'intention_to_migrate_legal', 'intention_to_migrate_illegal', 'Pays_etape', 'Nationalite_francaise', 'Mariage_blanc', 'Raisons_migration', 'Retour_pays', 'Proches_europe', 'Proche_fratrie', 'Proche_parent', 'Proche_ami', 'Proche_autre', 'Intention_envoyer_argent', 'Montant_envois', 'Frequence_envois']


## Étape 5 — Vérification des types

Inspection des `dtypes` et des valeurs uniques des colonnes à risque d'encodage (genre exporté en texte, codes pays).

In [6]:
print("Types après renommage :")
print(df_quest.dtypes)

print("\nValeurs uniques de Genre :")
print(df_quest["Genderf"].unique())

print("\nValeurs uniques de Pays_origine :")
print(df_quest["COUNRY_ID"].unique())

Types après renommage :
Age                             float64
Genderf                             str
COUNRY_ID                       float64
Regionf                         float64
Educationf                      float64
Social statusf                  float64
Professional situationf         float64
Taille_menage                   float64
Decideur_financier              float64
Revenu                          float64
Source_revenu_agriculture       float64
Source_revenu_commerce          float64
Source_revenu_salariat          float64
Source_revenu_aide_familiale    float64
Source_revenu_transferts        float64
Source_revenu_autre             float64
Bankaccount                     float64
Remit                           float64
Frequence_virements             float64
Choc_economique                 float64
Acces_credit                    float64
Corruption (0  1)               float64
GeneralizedTrsut                float64
TrustCourts                     float64
TrustLocalGovern

Recodage du genre (texte → binaire) avec `map`, pour obtenir une colonne numérique utilisable par le KNN et compatible avec la base d'enquête.

In [7]:
df_quest["Genderf"] = df_quest["Genderf"].map({"M": 1, "F": 0})

## Étape 6 — Ajout du libellé pays

L'export « codes » ne contient que l'identifiant numérique du pays. Le libellé est récupéré depuis l'export « réponses complètes » du même questionnaire : les deux fichiers ayant le même ordre de lignes, l'affectation est alignée sur l'index. Les libellés sont ensuite traduits par un dictionnaire de correspondance pour suivre la nomenclature (anglais, majuscules) de la base d'enquête.

In [8]:
df_quest_brut = pd.read_csv("../data/1_Donnees_brutes/base_questionnaire_brute.csv", encoding="utf-8")

# Récupérer la colonne pays
df_quest["Country"] = df_quest_brut["Quel est votre pays d'origine ?"]

# Dictionnaire de conversion FR -> EN
conversion_pays = {
    "ALBANIE": "ALBANIA",
    "ALGERIE": "ALGERIA",
    "BENIN": "BENIN",
    "BURKINA": "BURKINA FASO",
    "CENTRAFRICAINE (REPUBLIQUE)": "CENTRAL AFRICAN REPUBLIC",
    "COTE D'IVOIRE": "IVORY COAST",
    "EQUATEUR": "ECUADOR",
    "ESPAGNE": "SPAIN",
    "FRANCE": "FRANCE",
    "ITALIE": "ITALY",
    "KOSOVO": "KOSOVO",
    "LIBAN": "LEBANON",
    "MADAGASCAR": "MADAGASCAR",
    "MAROC": "MOROCCO",
    "MAURICE": "MAURITIUS",
    "SENEGAL": "SENEGAL",
    "TOGO": "TOGO",
    "TUNISIE": "TUNISIA",
    "TURQUIE": "TURKEY"
}

df_quest["Country"] = df_quest["Country"].map(conversion_pays)

# Vérification
print("Pays après conversion :")
print(df_quest["Country"].value_counts(dropna=False))

Pays après conversion :
Country
SPAIN           7
TUNISIA         7
SENEGAL         6
BENIN           4
BURKINA FASO    4
MAURITIUS       4
ALGERIA         4
ECUADOR         4
TOGO            4
TURKEY          3
KOSOVO          3
MADAGASCAR      3
ITALY           2
ALBANIA         2
LEBANON         2
MOROCCO         2
IVORY COAST     1
FRANCE          1
Name: count, dtype: int64


Filtrage du périmètre : exclusion, par masque booléen sur `Country`, des répondants dont le pays d'origine est hors du champ de l'étude.

In [9]:
df_quest = df_quest[df_quest['Country'] != 'FRANCE']

## Étape 7 — Imputation KNN

Toutes les colonnes numériques sont imputées par `KNNImputer` (la colonne texte `Country` est exclue du calcul). Le nombre de voisins est réduit (`n_neighbors=3`) par rapport à la base d'enquête car l'échantillon est petit : un voisinage large reviendrait à imputer par la moyenne globale. Les valeurs sont arrondies pour retrouver des codes de modalité valides.

In [10]:
df_pour_knn = df_quest.copy()
df_pour_knn = df_pour_knn.drop(columns=["Country"])

In [11]:
imputer = KNNImputer(n_neighbors=3, weights="distance")
test_imputer = imputer.fit_transform(df_pour_knn)

df_final = pd.DataFrame(test_imputer, columns=df_pour_knn.columns)
df_final = df_final.round().astype(int)

for col in df_final.columns:
    if col in df_final.columns:
        df_quest[col] = df_final[col].round().astype("int").values

df_quest.head()

,Age,Genderf,COUNRY_ID,Regionf,Educationf,Social statusf,Professional situationf,Taille_menage,Decideur_financier,Revenu,...,Retour_pays,Proches_europe,Proche_fratrie,Proche_parent,Proche_ami,Proche_autre,Intention_envoyer_argent,Montant_envois,Frequence_envois,Country
0,19,0,99015,2,2,5,7,2,1,1,...,1,0,1,1,1,1,0,3,2,TURKEY
1,27,0,99009,1,4,1,5,2,0,2,...,1,0,1,1,1,1,0,3,2,ITALY
2,40,1,99002,1,2,4,6,5,1,4,...,0,0,0,1,1,1,0,3,2,BENIN
3,44,0,99002,2,2,2,7,5,0,4,...,1,1,1,1,1,1,0,3,2,BENIN
4,26,0,99003,2,3,1,3,5,1,5,...,1,0,1,1,1,1,0,3,2,BURKINA FASO


## Étape 8 — Export

In [12]:
df_quest.to_excel("../data/2_Donnees_nettoyees/base_questionnaire_nettoyee.xlsx", index=False)